**Import bibliotek i utworzenie SparkSession**

Utworzono lokalną sesję Spark działającą w trybie local[*]. Wszystkie dostępne rdzenie procesora są wykorzystywane jako lokalny odpowiednik klastra Databricks.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FASTQ_Analysis")
    .master("local[*]")
    .getOrCreate()
)

spark

**7.1 Wczytanie i parsowanie pliku FASTQ**

Plik FASTQ został wczytany jako DataFrame tekstowy. Każda linia pliku stanowi osobny rekord. Ponieważ pojedynczy odczyt FASTQ składa się z 4 linii, liczba odczytów została wyznaczona jako liczba wszystkich linii podzielona przez 4. Plik SRR16356247_1_1.fastq powstał poprzez wyekstrahowanie pierwszych 100 odczytów z oryginalnego pliku: 
`head -n 400 SRR16356247_1.fastq > SRR16356247_1_1.fastq`

In [ ]:
file_path = "/home/clusters/work/SRR16356247_1_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

**7.2 Dodanie numerów linii**

Za pomocą funkcji `monotonically_increasing_id()` nadano każdemu wierszowi w DataFrame unikalny, rosnący numer. 

In [ ]:
lines_with_id = lines_df.withColumn(
    "line_number",
    monotonically_increasing_id()
)

lines_with_id.show(8, truncate=False)

In [ ]:
window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

**7.3 Określenie typu linii w FASTQ**

Przypisano etykiety kolejnym wierszom na podstawie ich funkcji.

In [ ]:
lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

**7.4 Utworzenie identyfikatora odczytu**

Nadanie wszysktim wierszom należacym do jednego rekordu tego samego `record_id`.

In [ ]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

**7.5 Przekształcenie do formatu szerokiego (Pivot)**

In [ ]:
fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)

fastq_wide.show(5, truncate=False)
fastq_wide.printSchema()

**7.6 Czyszczenie danych nagłówka**

In [ ]:
fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
).show(5, truncate=False)

In [ ]:
fastq_final = fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
)

fastq_final.cache()

fastq_final.count()

**Zadanie 2. Wstępna  analiza jakości**

In [ ]:
low_quality_reads = fastq_final.filter(
    col("quality").contains("#")
)

low_quality_reads.count()

**Porównajmy czas wykonania tego zadania z Zadaniem 1. Które było szybsze i
dlaczego? (podpowiedź: porównajmy złożoność operacji regexp_extract i length z
prostym contains).**

Szybsze było Zadanie 2 wykorzystujące contains(). Funkcja ta jedynie sprawdza, czy w tekście występuje określony fragment, dlatego jest prostą operacją. Natomiast w Zadaniu 1 Spark musiał dodatkowo wykonać regexp_extract(), aby wyodrębnić liczbę z nagłówka, oraz length(), aby obliczyć długość sekwencji. Są to bardziej złożone operacje, dlatego ich wykonanie zajmuje więcej czasu.

**Czy Spark wykorzystał cache? Przyjrzyjmy się liście stage'ów. Czy któryś z nich ma przy
sobie zieloną etykietę "skipped"? Jeśli tak, oznacza to, że Spark nie musiał ponownie
czytać danych z dysku, ponieważ skorzystał z wyniku zapisanego w pamięci przez
.cache() w poprzednim kroku.**

Tak, Spark wykorzystał cache. W Spark UI przy jednym ze Stage'ów widoczna była etykieta "skipped", co oznacza, że ten etap nie został wykonany ponownie. Zamiast ponownie odczytywać dane z dysku i wykonywać wcześniejsze obliczenia, Spark wykorzystał dane zapisane w pamięci.

**Jaki był Locality Level tasków? W szczegółach Stage'a znajdźmy informację o "Locality
Level". Czy dominowało PROCESS_LOCAL (najlepsza opcja, dane w pamięci tego
samego procesora)? Jeśli tak, dlaczego?**

Tak, dominowało PROCESS_LOCAL, ponieważ taski wykonyane były tam, gdzie dane były już dostępne w pamięci. W przypadku jedynego tasku w pochodzącego z jobu wywołanego w Zadaniu 2 było to możliwe, ponieważ DataFrame został wcześniej zapisany za pomocą .cache(), więc Spark nie musiał ponownie odczytywać danych z dysku ani przesyłać ich między executorami. Dzięki temu wykonanie było szybsze.

**Ile rekordów (Records) przetworzył każdy Task? Czy liczba ta jest zgodna z Twoimi
oczekiwaniami dotyczącymi podziału danych na partycje?**

Jedyny Task we fragmnecie kodu z Zadania 2 przetworzył 1 rekord. Jest to zgodne z oczekiwaniami, ponieważ po wykonaniu agregacji i wykorzystaniu danych z pamięci podręcznej (InMemoryTableScan) do etapu count() trafia już tylko jeden rekord zawierający wynik zliczania. Ponieważ DataFrame miał jedną partycję, Spark utworzył również tylko jeden Task. W kontraście taski w poprzednich Jobach przetwarzały po 400 rekordów.